# Brain tumor localization: train and evaluate on Google Colab

This notebook runs the project's existing scripts (`scripts/preprocess.py`, `scripts/train.py`, `scripts/evaluate.py`) on a Colab GPU. The model code is not duplicated here; the notebook only gets the code and data in place and calls the scripts.

**Before you start:** *Runtime → Change runtime type → GPU* (T4 or better).

**Storage layout:**
- **Code:** cloned from GitHub, or copied from your Google Drive.
- **Data:** the preprocessed slice cache (`data/processed`, about 100 MB) is copied from Drive to the Colab disk, which is fast. You can also download and preprocess the raw data here (step 3b).
- **Outputs:** checkpoints, logs, figures and metrics are written **straight to Drive**, so they survive a disconnect.

## 1. Settings

In [ ]:
# ---- edit these ------------------------------------------------------------------------------
CODE_SOURCE = "github"          # "github" = clone the repo | "drive" = copy the project folder from Drive
GITHUB_URL = "https://github.com/YuvalPartok/MRI_tumor_detection.git"
GITHUB_BRANCH = "main"

DRIVE_PROJECT = "/content/drive/MyDrive/MRI_tumor_identification"   # your project folder on Drive
DATA_SOURCE = "drive_processed" # "drive_processed" = copy data/processed from Drive (recommended)
                                # "download" = download raw data from TCIA and preprocess here (step 3b)
RUN_NAME = "colab_run1"         # outputs go to {DRIVE_PROJECT}/outputs/{RUN_NAME}

EPOCHS = None                   # None = use configs/config.yaml (max 60, early stopping); or e.g. 2 for a smoke test
NUM_WORKERS = 2                 # Colab has 2+ CPU cores; the data lives in memory, so 2 is plenty
# ------------------------------------------------------------------------------------------------
WORK_DIR = "/content/MRI_tumor_identification"   # local (fast) copy of the code + data

## 2. Mount Drive, check GPU, get the code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU! Runtime -> Change runtime type -> GPU"

In [ ]:
import os, shutil, subprocess
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
if CODE_SOURCE == "github":
    subprocess.run(["git", "clone", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_URL, WORK_DIR], check=True)
elif CODE_SOURCE == "drive":
    # Copy everything except the big data / output folders (they are handled below).
    shutil.copytree(DRIVE_PROJECT, WORK_DIR,
                    ignore=shutil.ignore_patterns("data", "outputs", ".venv", ".git", "__pycache__"))
else:
    raise ValueError(CODE_SOURCE)
os.chdir(WORK_DIR)
assert os.path.exists("src/mri_tumor/data/dataset.py"), "src/mri_tumor/data/ is missing from the code copy"
!git log -1 --oneline 2>/dev/null; ls

In [ ]:
# Colab already ships torch / torchvision / numpy / pandas / scipy / sklearn / matplotlib;
# this installs only what is missing (mainly nibabel) without touching the CUDA build of torch.
!pip install -q nibabel pyyaml tqdm
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 3a. Data: copy the preprocessed slices from Drive (recommended)

One-time setup: upload your local `data/processed/` folder (with `index.csv`, `splits.json` and `slices/`) to `{DRIVE_PROJECT}/data/processed/` on Drive. Using the same `splits.json` keeps the train/val/test patients **identical** to the local runs, so results stay comparable.

In [ ]:
if DATA_SOURCE == "drive_processed":
    src = f"{DRIVE_PROJECT}/data/processed"
    assert os.path.exists(f"{src}/splits.json"), f"upload data/processed to {src} first"
    shutil.copytree(src, f"{WORK_DIR}/data/processed", dirs_exist_ok=True)
    !du -sh data/processed; ls data/processed

## 3b. Data: download from TCIA and preprocess (alternative)

*Not tested on Colab yet.* TCIA serves the images only over IBM Aspera. This cell installs the Aspera CLI (`aspera-cli` needs Ruby ≥ 3.1, which is installed from conda-forge), downloads the FLAIR scan and tumor mask for `data.num_patients` patients (about 710 MB for 150), and preprocesses them. The raw volumes are also saved to Drive, so you only download them once.

In [ ]:
if DATA_SOURCE == "download":
    drive_raw = f"{DRIVE_PROJECT}/data/raw"
    os.makedirs(drive_raw, exist_ok=True)
    os.makedirs("data", exist_ok=True)
    if not os.path.exists("data/raw"):
        os.symlink(drive_raw, "data/raw")          # downloads land on Drive and persist
    # Ruby >= 3.1 via micromamba (Colab's apt Ruby is too old for aspera-cli)
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
    !micromamba create -y -q -p /opt/ruby -c conda-forge ruby=3.3 compilers make
    os.environ["PATH"] = "/opt/ruby/bin:" + os.environ["PATH"]
    !gem install aspera-cli --no-document
    !ascli config ascp install
    !python scripts/download_data.py
    !python scripts/preprocess.py

## 4. Colab config: outputs on Drive, Colab-friendly loader settings

In [ ]:
import yaml
OUT_DIR = f"{DRIVE_PROJECT}/outputs/{RUN_NAME}"
with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["paths"]["output_dir"] = OUT_DIR       # absolute path -> checkpoints survive a disconnect
cfg["train"]["num_workers"] = NUM_WORKERS
if EPOCHS is not None:
    cfg["train"]["epochs"] = EPOCHS
with open("configs/colab.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("outputs ->", OUT_DIR)
print({k: cfg["train"][k] for k in ["epochs", "scheduler", "monitor", "patience", "lr_patience", "batch_size", "num_workers"]})

## 5. Optional: sanity check of the ground-truth boxes

In [ ]:
!python scripts/visualize_samples.py --config configs/colab.yaml --augment
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"{OUT_DIR}/figures/sanity_samples*.png")):
    display(Image(p))

## 6. Train

This trains on the validation-driven schedule: LR × 0.3 when validation mean IoU stalls for 3 epochs, and early stopping after 10 epochs without improvement. The best checkpoint is saved to `{OUT_DIR}/checkpoints/best.pt`, and the full log to `{OUT_DIR}/logs/`.

In [ ]:
!python scripts/train.py --config configs/colab.yaml 2>&1 | tee /content/train_stdout.log
shutil.copy("/content/train_stdout.log", f"{OUT_DIR}/logs/train_stdout.log")

## 7. Evaluate on the held-out test patients

In [ ]:
!python scripts/evaluate.py --config configs/colab.yaml 2>&1 | tee /content/eval_stdout.log
shutil.copy("/content/eval_stdout.log", f"{OUT_DIR}/logs/eval_stdout.log")

## 8. Results

In [ ]:
import json, pandas as pd
m = json.load(open(f"{OUT_DIR}/metrics.json"))
print(f"checkpoint epoch: {m['checkpoint_epoch']} | test: {m['n_test_patients']} patients, {m['n_test_slices']} slices")
display(pd.Series(m["test"]).round(3).to_frame("test"))
if "test_by_tumor_size" in m:
    display(pd.DataFrame(m["test_by_tumor_size"]).T.round(3))
display(Image(f"{OUT_DIR}/figures/training_curves.png"))
for p in sorted(glob.glob(f"{OUT_DIR}/figures/test_predictions_*.png"))[:2]:
    display(Image(p))

Everything is in `{DRIVE_PROJECT}/outputs/{RUN_NAME}/` on Drive (`metrics.json`, `checkpoints/`, `logs/`, `figures/`). To use the figures in the README, copy `training_curves.png` and `test_predictions_1.png` / `_2.png` into `assets/`.